# Ausgrid monthly conditional VAE · Phase 2

This notebook extends the seasonal-resampling benchmark with a **conditional variational autoencoder** for 36-month household trajectories (January 2012–December 2014). It models base consumption, controlled load, and gross PV generation together. Generation is forced to zero for non-solar homes; a separate gate models months with controlled load.

**Run first:** `Ausgrid_Monthly_Data_Preparation_and_Baseline.ipynb`. This notebook reads its `monthly_work_package_1/` outputs with the **same household splits**. It writes checkpoints and evaluation results to `outputs/monthly_vae/`. In Colab, it searches both `MyDrive/ausgrid-synthetic/` and `MyDrive/ausgrid-synthetic/ausgrid-synthetic/`, mounting Drive only when needed. It also works from the local project root. Dependencies: NumPy, pandas, SciPy, scikit-learn, Matplotlib; no GPU or PyTorch is required.

The first benchmark used 4,514 training, 974 validation, and 969 held-out test households. The VAE generates profiles using static attributes only (cohort, LGA, and PV capacity); it never conditions on test-household consumption.


In [ ]:
from __future__ import annotations
import hashlib
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
from scipy.stats import spearmanr, wasserstein_distance
from sklearn.metrics import roc_auc_score

YEARS = (2012, 2013, 2014)
SEEDS = (11, 23, 37)
CHANNELS = ("base_kwh", "controlled_kwh", "generation_kwh")

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(4 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


## 1. Find the prepared monthly data in the project

The data checks are intentional: a folder with the same name but no prepared panel must not take precedence over the Drive project. The source panel, splits, and first baseline metrics remain unchanged.


In [ ]:
def resolve_project_root() -> Path:
    """Find prepared output locally or in one of the two prior Colab layouts."""
    workdir = Path.cwd().resolve()
    drive_root = Path("/content/drive/MyDrive/ausgrid-synthetic")
    candidates = [workdir, workdir.parent, workdir / "ausgrid-synthetic",
                  drive_root, drive_root / "ausgrid-synthetic"]

    def ready(p: Path) -> bool:
        parent = p / "monthly_work_package_1"
        return all((parent / f).is_file() for f in (
            "monthly_panel_2007_2014.csv.gz", "household_splits.csv", "baseline_metrics.csv"))

    root = next((p for p in candidates if ready(p)), None)
    if root is None:
        try:
            from google.colab import drive
        except ImportError as exc:
            raise FileNotFoundError("Run the first monthly notebook in ausgrid-synthetic before this one.") from exc
        drive.mount("/content/drive")
        root = next((p for p in (drive_root, drive_root / "ausgrid-synthetic") if ready(p)), None)
    if root is None:
        raise FileNotFoundError("First monthly notebook outputs not found in either MyDrive project layout.")
    return root.resolve()


def load_monthly_data(prepared_dir: Path) -> tuple[dict, dict, pd.DataFrame]:
    panel = pd.read_csv(prepared_dir / "monthly_panel_2007_2014.csv.gz", parse_dates=["month"])
    sites = pd.read_csv(prepared_dir / "household_splits.csv")
    baseline = pd.read_csv(prepared_dir / "baseline_metrics.csv")
    assert sites.household_id.is_unique and set(sites.split) == {"train", "validation", "test"}
    assert sites.overlap_interval.sum() == 204
    eligible = sites[sites.eligible_2012_2014].copy()
    assert eligible.groupby("split").size().to_dict() == {"train": 4514, "validation": 974, "test": 969}
    subset = panel[panel.household_id.isin(eligible.household_id) & panel.year.isin(YEARS)].copy()
    assert subset.groupby("household_id").size().eq(36).all()
    assert subset.usable_month.all()
    assert not subset.duplicated(["household_id", "month"]).any()

    lgas = sorted(eligible.loc[eligible.split == "train", "lga"].dropna().unique().tolist())
    lga_index = {name: i for i, name in enumerate(lgas)}
    result = {}
    for split in ("train", "validation", "test"):
        selected = eligible[eligible.split == split].sort_values("household_id").reset_index(drop=True)
        block = subset[subset.household_id.isin(selected.household_id)].sort_values(["household_id", "month"])
        assert block.household_id.drop_duplicates().tolist() == selected.household_id.tolist()
        raw = block[list(CHANNELS)].to_numpy(np.float32).reshape(len(selected), 36, 3)
        assert np.isfinite(raw).all() and (raw >= 0).all()
        cond = np.zeros((len(selected), 2 + len(lgas) + 1), dtype=np.float32)
        cond[:, 0] = selected.cohort.eq("solar").astype(np.float32)
        cond[:, 1] = selected.capacity_kw.fillna(0).to_numpy(np.float32) / 10.0
        for row, lga in enumerate(selected.lga):
            cond[row, 2 + lga_index.get(lga, len(lgas))] = 1.0
        result[split] = {"sites": selected, "raw": raw, "cond": cond,
                         "solar": selected.cohort.eq("solar").to_numpy(),
                         "controlled_present": raw[:, :, 1] > 0.01}
    log_train = np.log1p(result["train"]["raw"]).reshape(-1, 3)
    mean = log_train.mean(axis=0).astype(np.float32)
    std = np.maximum(log_train.std(axis=0), 0.1).astype(np.float32)
    for data in result.values():
        data["x"] = ((np.log1p(data["raw"]) - mean) / std).astype(np.float32)
    scaler = {"mean": mean, "std": std, "lgas": lgas}
    return result, scaler, baseline


In [ ]:
PROJECT_ROOT = resolve_project_root()
PREPARED_DIR = PROJECT_ROOT / "monthly_work_package_1"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "monthly_vae"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA, SCALER, BASELINE = load_monthly_data(PREPARED_DIR)
PANEL_PATH = PREPARED_DIR / "monthly_panel_2007_2014.csv.gz"
SPLITS_PATH = PREPARED_DIR / "household_splits.csv"
print("Project:", PROJECT_ROOT)
print("Prepared input:", PREPARED_DIR)
print("Output:", OUTPUT_DIR)
print("Eligible households:", {name: len(rows["sites"]) for name, rows in DATA.items()})
print("Panel SHA256:", sha256_file(PANEL_PATH))
print("Splits SHA256:", sha256_file(SPLITS_PATH))


## 2. VAE and gradient check

The encoder takes the observed 36 × 3 trajectory and static attributes. The decoder uses the latent variable and static attributes to generate the trajectory. Training uses a reparameterized Gaussian latent variable, reconstruction losses in scaled `log1p(kWh)` space, a controlled-load presence loss, and a warm-started KL term. The finite-difference check below compares analytic gradients with numerical gradients before training.


In [ ]:
class ConditionalVAE:
    """One-hidden-layer conditional VAE with an explicit controlled-load gate."""

    def __init__(self, condition_dim: int, seed: int, hidden: int = 128,
                 latent: int = 16, dtype=np.float32):
        rng = np.random.default_rng(seed)
        self.condition_dim, self.hidden, self.latent = condition_dim, hidden, latent
        self.dtype = dtype

        def weight(m, n):
            return (rng.standard_normal((m, n)) * np.sqrt(2.0 / (m + n))).astype(dtype)

        self.params = {
            "We": weight(108 + condition_dim, hidden), "be": np.zeros(hidden, dtype),
            "Wmu": weight(hidden, latent), "bmu": np.zeros(latent, dtype),
            "Wlv": weight(hidden, latent), "blv": np.zeros(latent, dtype),
            "Wd": weight(latent + condition_dim, hidden), "bd": np.zeros(hidden, dtype),
            "Wo": weight(hidden, 108 + 36), "bo": np.zeros(108 + 36, dtype),
        }

    def decode(self, z: np.ndarray, cond: np.ndarray):
        p = self.params
        h = np.tanh(np.concatenate([z, cond], axis=1) @ p["Wd"] + p["bd"])
        out = h @ p["Wo"] + p["bo"]
        return out[:, :108].reshape(-1, 36, 3), out[:, 108:], h

    def loss_and_grad(self, x: np.ndarray, cond: np.ndarray, solar: np.ndarray,
                      ctl: np.ndarray, beta: float, eps: np.ndarray):
        p = self.params
        b = x.shape[0]
        enc_input = np.concatenate([x.reshape(b, 108), cond], axis=1)
        he = np.tanh(enc_input @ p["We"] + p["be"])
        mu = he @ p["Wmu"] + p["bmu"]
        lv_raw = he @ p["Wlv"] + p["blv"]
        lv = np.clip(lv_raw, -6.0, 4.0)
        sd = np.exp(0.5 * lv)
        z = mu + sd * eps
        y, logits, hd = self.decode(z, cond)
        diff = y - x
        ctl_f = ctl.astype(x.dtype)
        solar_f = solar.astype(x.dtype)[:, None]
        gate_weight = 0.3

        base = np.mean(diff[:, :, 0] ** 2)
        controlled = np.sum((diff[:, :, 1] ** 2) * ctl_f) / max(float(ctl_f.sum()), 1.0)
        generation = np.sum((diff[:, :, 2] ** 2) * solar_f) / max(float(solar_f.sum() * 36), 1.0)
        gate = np.mean(np.logaddexp(0.0, logits) - ctl_f * logits)
        kl = 0.5 * np.mean(np.sum(mu ** 2 + np.exp(lv) - 1.0 - lv, axis=1))
        total = float(base + controlled + generation + gate_weight * gate + beta * kl)

        dy = np.zeros_like(y)
        dy[:, :, 0] = 2.0 * diff[:, :, 0] / (b * 36)
        dy[:, :, 1] = 2.0 * diff[:, :, 1] * ctl_f / max(float(ctl_f.sum()), 1.0)
        dy[:, :, 2] = 2.0 * diff[:, :, 2] * solar_f / max(float(solar_f.sum() * 36), 1.0)
        sigmoid = 1.0 / (1.0 + np.exp(-np.clip(logits, -30, 30)))
        dlogits = gate_weight * (sigmoid - ctl_f) / (b * 36)
        dout = np.concatenate([dy.reshape(b, 108), dlogits], axis=1)

        grad = {}
        grad["Wo"] = hd.T @ dout
        grad["bo"] = dout.sum(axis=0)
        dhd = (dout @ p["Wo"].T) * (1.0 - hd ** 2)
        dec_in = np.concatenate([z, cond], axis=1)
        grad["Wd"] = dec_in.T @ dhd
        grad["bd"] = dhd.sum(axis=0)
        dz = (dhd @ p["Wd"].T)[:, :self.latent]
        dmu = dz + beta * mu / b
        dlv = dz * (0.5 * sd * eps) + beta * 0.5 * (np.exp(lv) - 1.0) / b
        dlv *= ((lv_raw > -6.0) & (lv_raw < 4.0))
        grad["Wmu"] = he.T @ dmu
        grad["bmu"] = dmu.sum(axis=0)
        grad["Wlv"] = he.T @ dlv
        grad["blv"] = dlv.sum(axis=0)
        dhe = (dmu @ p["Wmu"].T + dlv @ p["Wlv"].T) * (1.0 - he ** 2)
        grad["We"] = enc_input.T @ dhe
        grad["be"] = dhe.sum(axis=0)
        return total, grad, {"base": float(base), "controlled": float(controlled),
                             "generation": float(generation), "gate": float(gate), "kl": float(kl)}


def check_gradients() -> dict:
    rng = np.random.default_rng(123)
    model = ConditionalVAE(condition_dim=5, seed=7, hidden=6, latent=3, dtype=np.float64)
    x = rng.normal(size=(4, 36, 3)).astype(np.float64)
    cond = rng.normal(size=(4, 5)).astype(np.float64)
    solar = np.array([True, False, True, False])
    ctl = rng.random((4, 36)) < 0.4
    eps = rng.normal(size=(4, 3)).astype(np.float64)
    _, grads, _ = model.loss_and_grad(x, cond, solar, ctl, beta=0.03, eps=eps)
    checks = {}
    for key, index in [("We", (0, 0)), ("Wmu", (0, 1)), ("Wlv", (1, 2)),
                       ("Wd", (0, 2)), ("Wo", (2, 0)), ("Wo", (2, 110)), ("bo", (109,))]:
        param = model.params[key]
        old = float(param[index])
        delta = 1e-5
        param[index] = old + delta
        hi = model.loss_and_grad(x, cond, solar, ctl, beta=0.03, eps=eps)[0]
        param[index] = old - delta
        lo = model.loss_and_grad(x, cond, solar, ctl, beta=0.03, eps=eps)[0]
        param[index] = old
        numeric = (hi - lo) / (2 * delta)
        analytic = float(grads[key][index])
        error = abs(numeric - analytic) / max(1e-5, abs(numeric), abs(analytic))
        checks[f"{key}_{index}"] = error
        assert error < 5e-4, (key, index, analytic, numeric, error)
    return checks


In [ ]:
GRADIENT_ERRORS = check_gradients()
print("Gradient check passed; maximum relative error:", max(GRADIENT_ERRORS.values()))


## 3. Train on fixed training households

The validation set selects the best epoch independently for each of three seeds. The test set is untouched until the next section. Checkpoints and scaler metadata are saved so the generated samples can be reproduced. The NumPy implementation runs on CPU; expect longer runtime than the preparation notebook.


In [ ]:
def train_one_seed(data: dict, seed: int, max_epochs: int = 120, patience: int = 18,
                   batch_size: int = 256, hidden: int = 128, latent: int = 16,
                   learning_rate: float = 0.001) -> tuple[ConditionalVAE, pd.DataFrame]:
    rng = np.random.default_rng(seed)
    train, val = data["train"], data["validation"]
    model = ConditionalVAE(train["cond"].shape[1], seed, hidden, latent)
    m = {k: np.zeros_like(v) for k, v in model.params.items()}
    v = {k: np.zeros_like(w) for k, w in model.params.items()}
    step = 0
    best, bad, best_params, history = np.inf, 0, None, []
    for epoch in range(1, max_epochs + 1):
        beta = 0.03 * min(1.0, epoch / 20.0)
        training_losses = []
        for indices in np.array_split(rng.permutation(len(train["x"])),
                                      int(np.ceil(len(train["x"]) / batch_size))):
            eps = rng.standard_normal((len(indices), latent)).astype(np.float32)
            batch_loss, grad, _ = model.loss_and_grad(train["x"][indices], train["cond"][indices],
                                                       train["solar"][indices], train["controlled_present"][indices],
                                                       beta=beta, eps=eps)
            training_losses.append(batch_loss)
            norm = np.sqrt(sum(float(np.sum(g * g)) for g in grad.values()))
            if norm > 5:
                grad = {k: g * (5.0 / norm) for k, g in grad.items()}
            step += 1
            for key, g in grad.items():
                m[key] = 0.9 * m[key] + 0.1 * g
                v[key] = 0.999 * v[key] + 0.001 * (g * g)
                model.params[key] -= learning_rate * (m[key] / (1 - 0.9 ** step)) / (
                    np.sqrt(v[key] / (1 - 0.999 ** step)) + 1e-8)
        # Fixed validation noise makes early stopping comparisons stable.
        vrng = np.random.default_rng(9000 + seed)
        val_eps = vrng.standard_normal((len(val["x"]), latent)).astype(np.float32)
        val_loss, _, parts = model.loss_and_grad(val["x"], val["cond"], val["solar"],
                                                val["controlled_present"], beta=0.03, eps=val_eps)
        history.append({"seed": seed, "epoch": epoch, "training_loss": float(np.mean(training_losses)),
                        "validation_loss": val_loss, **parts})
        if np.isfinite(val_loss) and val_loss < best - 1e-4:
            best, bad = val_loss, 0
            best_params = {k: x.copy() for k, x in model.params.items()}
        else:
            bad += 1
        if epoch % 10 == 0 or epoch == 1 or bad >= patience:
            print(f"seed={seed} epoch={epoch} val_loss={val_loss:.4f} best={best:.4f}")
        if bad >= patience:
            break
    assert best_params is not None
    model.params = best_params
    return model, pd.DataFrame(history)


In [ ]:
MAX_EPOCHS = 120
PATIENCE = 18
BATCH_SIZE = 256
HIDDEN = 128
LATENT = 16
LEARNING_RATE = 0.001

MODELS, HISTORIES, DURATIONS = {}, [], {}
for seed in SEEDS:
    start = time.perf_counter()
    model, history = train_one_seed(
        DATA, seed, max_epochs=MAX_EPOCHS, patience=PATIENCE,
        batch_size=BATCH_SIZE, hidden=HIDDEN, latent=LATENT,
        learning_rate=LEARNING_RATE)
    DURATIONS[str(seed)] = round(time.perf_counter() - start, 1)
    MODELS[seed] = model
    HISTORIES.append(history)
    np.savez_compressed(OUTPUT_DIR / f"conditional_vae_seed{seed}.npz", **model.params)
    best_row = history.loc[history.validation_loss.idxmin()]
    print(f"seed={seed}: best epoch={int(best_row.epoch)}, best val loss={best_row.validation_loss:.4f}, time={DURATIONS[str(seed)]}s")
pd.concat(HISTORIES, ignore_index=True).to_csv(OUTPUT_DIR / "training_history.csv", index=False)
config = {"seeds": SEEDS, "years": YEARS, "channels": CHANNELS,
          "max_epochs": MAX_EPOCHS, "patience": PATIENCE, "batch_size": BATCH_SIZE,
          "hidden": HIDDEN, "latent": LATENT, "learning_rate": LEARNING_RATE,
          "kl_beta_final": 0.03, "controlled_gate_loss_weight": 0.3,
          "input_panel_sha256": sha256_file(PANEL_PATH),
          "input_splits_sha256": sha256_file(SPLITS_PATH),
          "scaler_mean_log1p_kwh": SCALER["mean"].tolist(),
          "scaler_std_log1p_kwh": SCALER["std"].tolist(),
          "training_lga_vocabulary": SCALER["lgas"], "training_seconds": DURATIONS}
(OUTPUT_DIR / "config.json").write_text(json.dumps(config, indent=2) + "\n")


## 4. Generate from held-out attributes and evaluate

For each seed, sample one independent profile per eligible held-out household. Evaluate the same annual and monthly Wasserstein metrics and 2014 relationships as the seasonal benchmark. A separate nearest-neighbor membership probe uses a synthetic pool generated from *training attributes only*. It is a single privacy attack; an AUC near 0.5 does not certify privacy.

The output CSV uses synthetic IDs. Test household IDs are used internally for paired evaluation but omitted from the saved generated file.


In [ ]:
def sample_profiles(model: ConditionalVAE, data: dict, scaler: dict, seed: int) -> pd.DataFrame:
    rng = np.random.default_rng(seed + 100_000)
    sites = data["sites"]
    z = rng.standard_normal((len(sites), model.latent)).astype(np.float32)
    predicted, logits, _ = model.decode(z, data["cond"])
    log_kwh = predicted * scaler["std"][None, None, :] + scaler["mean"][None, None, :]
    values = np.expm1(np.clip(log_kwh, 0, np.log1p(50_000))).astype(np.float32)
    gate_prob = 1.0 / (1.0 + np.exp(-np.clip(logits, -30, 30)))
    values[:, :, 1] *= rng.random(gate_prob.shape) < gate_prob
    values[~data["solar"], :, 2] = 0.0
    months = pd.date_range("2012-01-01", "2014-12-01", freq="MS")
    capacity = sites.capacity_kw.fillna(0).to_numpy(np.float32)
    upper = capacity[:, None] * (months.days_in_month.to_numpy()[None, :] * 24)
    values[:, :, 2] = np.minimum(values[:, :, 2], upper)
    anonymous = rng.integers(0, 2**63, size=len(sites), dtype=np.int64)
    assert len(np.unique(anonymous)) == len(sites)
    frame = pd.DataFrame({
        "household_id": np.repeat(sites.household_id.to_numpy(), 36),
        "synthetic_id": np.repeat([f"vae_{seed}_{int(v):016x}" for v in anonymous], 36),
        "cohort": np.repeat(sites.cohort.to_numpy(), 36),
        "lga": np.repeat(sites.lga.to_numpy(), 36),
        "capacity_kw": np.repeat(sites.capacity_kw.to_numpy(), 36),
        "month": np.tile(months.to_numpy(), len(sites)),
        "base_kwh": values[:, :, 0].reshape(-1),
        "controlled_kwh": values[:, :, 1].reshape(-1),
        "generation_kwh": values[:, :, 2].reshape(-1),
    })
    frame["year"] = frame.month.dt.year
    frame["month_number"] = frame.month.dt.month
    frame["total_consumption_kwh"] = frame.base_kwh + frame.controlled_kwh
    # Shuffle whole trajectories so file order does not expose the sorted
    # test-household order used by the fixed split manifest.
    order = np.concatenate([np.arange(i * 36, (i + 1) * 36) for i in rng.permutation(len(sites))])
    return frame.iloc[order].reset_index(drop=True)


def marginal_metrics(real: pd.DataFrame, synthetic: pd.DataFrame, seed: int) -> pd.DataFrame:
    rows = []
    for cohort in ("solar", "non-solar"):
        r = real[real.cohort == cohort]
        s = synthetic[synthetic.cohort == cohort]
        for channel in ("total_consumption_kwh", "controlled_kwh", "generation_kwh"):
            if cohort == "non-solar" and channel == "generation_kwh":
                continue
            a = r.groupby(["household_id", "year"])[channel].sum().to_numpy()
            b = s.groupby(["household_id", "year"])[channel].sum().to_numpy()
            w1 = wasserstein_distance(a, b)
            med = np.median(a)
            month_w1 = np.mean([wasserstein_distance(r.loc[r.month_number == m, channel],
                                                       s.loc[s.month_number == m, channel]) for m in range(1, 13)])
            rows.append({"seed": seed, "cohort": cohort, "channel": channel,
                         "target_households": int(r.household_id.nunique()),
                         "annual_real_median_kwh": float(med),
                         "annual_synthetic_median_kwh": float(np.median(b)),
                         "annual_w1_kwh": float(w1),
                         "annual_w1_over_real_median": float(w1 / med) if med > 0 else np.nan,
                         "mean_calendar_month_w1_kwh": float(month_w1),
                         "zero_month_rate_real": float((r[channel] == 0).mean()),
                         "zero_month_rate_synthetic": float((s[channel] == 0).mean())})
    return pd.DataFrame(rows)


def relationship_metrics(real: pd.DataFrame, synthetic: pd.DataFrame, seed: int) -> pd.DataFrame:
    rows = []
    for name, df in (("real", real), ("synthetic", synthetic)):
        for cohort in ("solar", "non-solar"):
            annual = df[(df.cohort == cohort) & (df.year == 2014)].groupby("household_id").agg(
                consumption=("total_consumption_kwh", "sum"),
                controlled=("controlled_kwh", "sum"),
                generation=("generation_kwh", "sum"),
                capacity=("capacity_kw", "first"))
            rows.append({"seed": seed, "kind": name, "cohort": cohort, "households": len(annual),
                         "controlled_load_presence_fraction": float((annual.controlled > 0).mean()),
                         "capacity_generation_spearman": float(spearmanr(annual.capacity, annual.generation).statistic) if cohort == "solar" else np.nan,
                         "consumption_generation_spearman": float(spearmanr(annual.consumption, annual.generation).statistic) if cohort == "solar" else np.nan})
    return pd.DataFrame(rows)


def privacy_probe(model: ConditionalVAE, data: dict, scaler: dict, seed: int) -> pd.DataFrame:
    """Nearest-synthetic neighbor membership probe, stratified by cohort.

    Generate a separate pool from training *attributes* only, then compare
    equally sized train and test profiles with the same distance scoring rule.
    This probe is limited; an AUC near 0.5 is no privacy guarantee.
    """
    rng = np.random.default_rng(seed + 50_000)
    rows = []
    for cohort in ("solar", "non-solar"):
        is_solar = cohort == "solar"
        tr = np.flatnonzero(data["train"]["solar"] == is_solar)
        te = np.flatnonzero(data["test"]["solar"] == is_solar)
        chosen = rng.choice(tr, len(te), replace=False)
        conditions = {key: (value.iloc[chosen].reset_index(drop=True) if key == "sites"
                            else value[chosen]) for key, value in data["train"].items()}
        pool = sample_profiles(model, conditions, scaler, seed + (1 if is_solar else 2))
        pool_raw = pool[list(CHANNELS)].to_numpy(np.float32).reshape(len(te), 36, 3)
        channels = (0, 1, 2) if is_solar else (0, 1)
        def features(raw):
            transformed = (np.log1p(raw) - scaler["mean"]) / scaler["std"]
            return transformed[:, :, channels].reshape(len(raw), -1).astype(np.float64)
        ref = features(pool_raw)
        train_dist = cdist(features(data["train"]["raw"][chosen]), ref).min(axis=1)
        test_dist = cdist(features(data["test"]["raw"][te]), ref).min(axis=1)
        auc = roc_auc_score(np.r_[np.ones(len(te)), np.zeros(len(te))],
                            -np.r_[train_dist, test_dist])
        rows.append({"seed": seed, "cohort": cohort, "train_cases": len(te), "test_cases": len(te),
                     "nearest_neighbor_attack_auc": float(auc),
                     "train_median_distance": float(np.median(train_dist)),
                     "test_median_distance": float(np.median(test_dist))})
    return pd.DataFrame(rows)


def plot_comparison(real: pd.DataFrame, baseline: pd.DataFrame, synthetic: pd.DataFrame,
                    path: Path) -> None:
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), constrained_layout=True)
    cases = [("solar", "total_consumption_kwh", "Solar consumption"),
             ("solar", "generation_kwh", "Solar generation"),
             ("non-solar", "total_consumption_kwh", "Non-solar consumption")]
    for ax, (cohort, channel, title) in zip(axes, cases):
        for name, df, style in (("Held-out real", real, "-"),
                                ("Resampling", baseline, ":"), ("Conditional VAE", synthetic, "--")):
            series = df[(df.cohort == cohort) & (df.year == 2014)].groupby("month_number")[channel].median()
            ax.plot(series.index, series.to_numpy(), style, linewidth=2, label=name)
        ax.set(title=title, xlabel="Month of 2014", ylabel="Median kWh per household")
        ax.set_xticks([1, 3, 5, 7, 9, 11])
        ax.grid(alpha=0.2)
    axes[0].legend(frameon=False)
    fig.savefig(path, dpi=175)
    plt.close(fig)


In [ ]:
real = pd.read_csv(PANEL_PATH, parse_dates=["month"])
real = real[real.household_id.isin(DATA["test"]["sites"].household_id) & real.year.isin(YEARS)].copy()
baseline_sample = pd.read_csv(PREPARED_DIR / "baseline_synthetic_2012_2014_seed11.csv.gz", parse_dates=["month"])
METRIC_TABLES, RELATIONSHIP_TABLES, PRIVACY_TABLES = [], [], []
for seed in SEEDS:
    generated = sample_profiles(MODELS[seed], DATA["test"], SCALER, seed)
    assert len(generated) == len(DATA["test"]["sites"]) * 36
    assert np.isfinite(generated[list(CHANNELS)]).all().all()
    assert generated[list(CHANNELS)].ge(0).all().all()
    assert generated.loc[generated.cohort == "non-solar", "generation_kwh"].eq(0).all()
    METRIC_TABLES.append(marginal_metrics(real, generated, seed))
    RELATIONSHIP_TABLES.append(relationship_metrics(real, generated, seed))
    PRIVACY_TABLES.append(privacy_probe(MODELS[seed], DATA, SCALER, seed))
    if seed == SEEDS[0]:
        generated.drop(columns="household_id").to_csv(
            OUTPUT_DIR / "synthetic_test_seed11.csv.gz", index=False, compression="gzip")
        plot_comparison(real, baseline_sample, generated, OUTPUT_DIR / "seasonality_comparison_2014.png")

metrics = pd.concat(METRIC_TABLES, ignore_index=True)
relationships = pd.concat(RELATIONSHIP_TABLES, ignore_index=True)
privacy = pd.concat(PRIVACY_TABLES, ignore_index=True)
metrics.to_csv(OUTPUT_DIR / "vae_metrics.csv", index=False)
relationships.to_csv(OUTPUT_DIR / "vae_relationship_metrics.csv", index=False)
privacy.to_csv(OUTPUT_DIR / "nearest_neighbor_membership_probe.csv", index=False)

comparison = (metrics.groupby(["cohort", "channel"])["annual_w1_over_real_median"].mean().rename("vae")
              .to_frame().join(BASELINE.groupby(["cohort", "channel"])["annual_w1_over_real_median"]
                               .mean().rename("resampling")))
comparison["vae_minus_resampling"] = comparison.vae - comparison.resampling
comparison.to_csv(OUTPUT_DIR / "comparison_to_resampling.csv")
summary = {"mean_normalized_annual_w1_by_channel": comparison.reset_index().replace({np.nan: None}).to_dict("records"),
           "membership_auc_by_cohort": privacy.groupby("cohort").nearest_neighbor_attack_auc.agg(["mean", "min", "max"]).to_dict("index"),
           "note": "Nearest-neighbor membership AUC is one limited attack, not a privacy guarantee."}
(OUTPUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2) + "\n")
print(comparison.round(3).to_string())
print("\nMembership probe:\n", privacy.round(3).to_string(index=False))


## 5. Review and decide

Inspect the seasonal plot and the relationship metrics alongside the marginal score table. The resampling baseline retains donor profile shapes; the VAE should be assessed for fidelity, diversity, and disclosure together. No synthetic output is declared safe for release by this notebook. The 2011–2013 half-hour overlap experiment remains a **separate** benchmark because none of those 204 households has a complete 2012–2014 monthly record.


In [ ]:
from IPython.display import display, Image
display(Image(filename=str(OUTPUT_DIR / "seasonality_comparison_2014.png")))
print("Saved files:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(" ", path.name)
